# EU Electricity Prices - CSV Data Cleaning
## Using Pure Python File I/O with `with` Statements

This notebook demonstrates:
- Reading CSV files using `with` statement
- Processing and cleaning data with built-in Python
- Writing cleaned data back to CSV
- No external libraries (pandas/numpy) - pure file I/O approach

## Step 1: Create Sample Data
First, let's create a sample EU electricity prices CSV with some common data quality issues.

In [ ]:
# Create sample raw data file with common issues
sample_data = """Date,Country,Price_EUR_MWh,Volume_MWh,Status
2024-01-01,Germany,85.50,1200,confirmed
2024-01-01,France,78.25,1500,confirmed
2024-01-01,Germany,85.50,1200,confirmed
2024-01-02,Spain, 92.10 ,950,preliminary
2024-01-02,Italy,88.75,,confirmed
2024-01-03,Germany,,1100,confirmed
2024-01-03,Netherlands,81.30,1050,confirmed
2024-01-03,Belgium,83.20,1200,CONFIRMED
2024-01-04,Poland,79.45,900,preliminary
2024-01-04,France,76.80,1400,confirmed
2024-01-04,,88.15,1050,confirmed
"""

# Write sample data to file
with open('raw_electricity_prices.csv', 'w') as f:
    f.write(sample_data)

print("✓ Sample data file created: raw_electricity_prices.csv")
print("\nSample data preview:")
print(sample_data)

## Step 2: Define Data Cleaning Functions
Pure Python functions to handle various cleaning tasks.

In [ ]:
def parse_csv_line(line):
    """Parse a CSV line into fields, handling quoted values."""
    fields = []
    current_field = ""
    in_quotes = False
    
    for char in line:
        if char == '"':
            in_quotes = not in_quotes
        elif char == ',' and not in_quotes:
            fields.append(current_field)
            current_field = ""
        else:
            current_field += char
    
    fields.append(current_field)
    return fields

def clean_field(value):
    """Clean a field: strip whitespace and handle empty values."""
    cleaned = value.strip()
    return cleaned if cleaned else None

def validate_row(row, headers):
    """Validate and return row, or None if invalid."""
    # Check for duplicate rows (compare date + country + price)
    if row['Date'] and row['Country'] and row['Price_EUR_MWh']:
        # Validate numeric fields
        try:
            if row['Price_EUR_MWh']:
                float(row['Price_EUR_MWh'])
            if row['Volume_MWh'] and row['Volume_MWh'] != 'None':
                float(row['Volume_MWh'])
            return row
        except ValueError:
            return None
    return None

print("✓ Cleaning functions defined")

## Step 3: Read, Clean, and Deduplicate Data
Main processing logic using file I/O with `with` statements.

In [ ]:
# Read and process the CSV file
headers = None
rows = []
seen_rows = set()  # For deduplication
errors = []  # Track problematic rows

with open('raw_electricity_prices.csv', 'r') as input_file:
    for line_num, line in enumerate(input_file, 1):
        line = line.rstrip('\n\r')
        
        # Skip empty lines
        if not line.strip():
            continue
        
        # Parse CSV line
        fields = parse_csv_line(line)
        
        # First line: headers
        if line_num == 1:
            headers = [clean_field(f) for f in fields]
            print(f"Headers: {headers}")
            continue
        
        # Process data row
        if len(fields) == len(headers):
            # Create dictionary from row
            row_dict = {}
            for i, header in enumerate(headers):
                cleaned_value = clean_field(fields[i]) if i < len(fields) else None
                row_dict[header] = cleaned_value
            
            # Standardize status field (make uppercase consistent)
            if row_dict.get('Status'):
                row_dict['Status'] = row_dict['Status'].upper()
            
            # Validate row
            validated_row = validate_row(row_dict, headers)
            
            if validated_row:
                # Create deduplication key (date + country + price)
                dup_key = (validated_row['Date'], validated_row['Country'], validated_row['Price_EUR_MWh'])
                
                if dup_key not in seen_rows:
                    seen_rows.add(dup_key)
                    rows.append(validated_row)
                else:
                    errors.append(f"Line {line_num}: Duplicate row - {validated_row}")
            else:
                errors.append(f"Line {line_num}: Missing required fields or invalid numeric values")
        else:
            errors.append(f"Line {line_num}: Field count mismatch (expected {len(headers)}, got {len(fields)})")

print(f"\n✓ Read {len(rows)} valid rows")
print(f"✗ Skipped {len(errors)} problematic rows")

if errors:
    print("\nCleaning Report:")
    for error in errors:
        print(f"  {error}")

## Step 4: Display Cleaned Data Summary

In [ ]:
print("\n" + "="*80)
print("CLEANED DATA PREVIEW")
print("="*80)

if rows:
    # Display header
    print(f"\n{','.join(headers)}")
    
    # Display rows
    for row in rows:
        values = [str(row.get(h, '')) for h in headers]
        print(','.join(values))
    
    print(f"\nTotal cleaned records: {len(rows)}")
    
    # Basic statistics
    prices = []
    volumes = []
    countries = set()
    dates = set()
    
    for row in rows:
        try:
            if row['Price_EUR_MWh']:
                prices.append(float(row['Price_EUR_MWh']))
            if row['Volume_MWh'] and row['Volume_MWh'] != 'None':
                volumes.append(float(row['Volume_MWh']))
            if row['Country']:
                countries.add(row['Country'])
            if row['Date']:
                dates.add(row['Date'])
        except ValueError:
            pass
    
    print(f"\nData Statistics:")
    print(f"  - Countries: {len(countries)} ({', '.join(sorted(countries))})")
    print(f"  - Date range: {min(dates)} to {max(dates)}" if dates else "  - Date range: N/A")
    if prices:
        print(f"  - Price range: €{min(prices):.2f} - €{max(prices):.2f} per MWh")
        print(f"  - Average price: €{sum(prices)/len(prices):.2f} per MWh")
    if volumes:
        print(f"  - Volume range: {min(volumes):.0f} - {max(volumes):.0f} MWh")
        print(f"  - Total volume: {sum(volumes):.0f} MWh")

## Step 5: Write Cleaned Data to New CSV File
Using `with` statement for safe file writing.

In [ ]:
# Write cleaned data to output file
output_filename = 'cleaned_electricity_prices.csv'

with open(output_filename, 'w') as output_file:
    # Write header
    output_file.write(','.join(headers) + '\n')
    
    # Write data rows
    for row in rows:
        values = [str(row.get(h, '')) for h in headers]
        output_file.write(','.join(values) + '\n')

print(f"✓ Cleaned data written to: {output_filename}")
print(f"  - Records: {len(rows)}")
print(f"  - File size: approximately {sum(len(str(row.get(h, '')) for h in headers)) for row in rows)} bytes")

## Step 6: Verify Output File

In [ ]:
# Read and display the output file to verify
print(f"\nVerifying output file: {output_filename}")
print("="*80)

with open(output_filename, 'r') as verify_file:
    line_count = 0
    for line in verify_file:
        print(line.rstrip())
        line_count += 1

print("="*80)
print(f"✓ File verified: {line_count} lines (1 header + {line_count-1} data rows)")

## Summary of Cleaning Operations

### Issues Handled:
1. **Whitespace**: Trimmed leading/trailing spaces in all fields
2. **Duplicates**: Removed duplicate rows (same date, country, price)
3. **Missing Values**: Excluded rows with missing required fields (Date, Country, Price)
4. **Data Type Validation**: Validated numeric fields (Price, Volume)
5. **Standardization**: Normalized Status field to uppercase
6. **Malformed Rows**: Filtered out rows with field count mismatches

### Key Python Features Used:
- **`with` statements** for safe file I/O
- **List comprehensions** for data transformation
- **Dictionary operations** for row processing
- **Set operations** for deduplication
- **Exception handling** for data validation
- **String methods** for data cleaning